# 金融智能体 Demo：马科维茨投资组合优化（A 股 / 美股）

**课程：《金融智能体设计》** | 教学案例

## 架构概览

```
用户输入（股票列表 + 市场选择）
    ↓
DeepSeek 大模型（LangChain Agent）
    ├─ 工具1: download_stock_data  → A股(akshare) 或 美股(yfinance)
    ├─ 工具2: markowitz_optimize   → scipy 马科维茨最优化
    └─ 最终回答：最优权重 + 风险指标 + 配置建议
```

## 主要依赖
| 库 | 用途 |
|---|---|
| `langchain` + `langchain-deepseek` | 智能体框架 + DeepSeek 模型接入 |
| `akshare` | A 股免费数据源（国内访问） |
| `yfinance` | 美股数据源（境外访问） |
| `scipy` + `numpy` | 马科维茨数值优化 |
| `python-dotenv` | 读取 .env 中的 API Key |


In [7]:
import os
import json
import numpy as np
import pandas as pd
import akshare as ak
import yfinance as yf
from scipy.optimize import minimize
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent

print('所有依赖导入成功')


所有依赖导入成功


## 第一步：初始化 DeepSeek 大模型

从 `.env` 文件读取 `DEEPSEEK_API_KEY`，创建 `ChatDeepSeek` 实例。


In [8]:
load_dotenv()

llm = ChatDeepSeek(
    model='deepseek-chat',
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    temperature=0,
)

print('LLM 初始化成功，模型：', llm.model_name)


LLM 初始化成功，模型： deepseek-chat


## 第二步：定义工具（Tools）

这里定义两个工具：

1. **`download_stock_data`**：支持 A 股（akshare）和美股（yfinance）两个数据源，通过 `market` 参数切换
2. **`markowitz_optimize`**：对下载数据做马科维茨优化，求最大夏普比率组合与最小方差组合

> `market='A'` → akshare（适合国内网络）
> `market='US'` → yfinance（适合境外网络）


In [9]:
_portfolio_data: dict = {}

@tool
def download_stock_data(
    stock_codes: str,
    market: str = 'A',
    start_date: str = '20240101',
    end_date: str = '20241231',
) -> str:
    '''下载股票历史行情数据并计算日收益率，支持 A 股和美股。

    Args:
        stock_codes: 股票代码，多只用英文逗号分隔。
                     A股示例：000001,600519,300750
                     美股示例：AAPL,MSFT,GOOGL
        market: 市场选择，A 表示 A 股，US 表示美股，默认 A
        start_date: 起始日期，格式 YYYYMMDD，默认 20240101
        end_date: 结束日期，格式 YYYYMMDD，默认 20241231

    Returns:
        数据下载状态及基本统计摘要（JSON 字符串）
    '''
    codes = [c.strip() for c in stock_codes.split(',')]

    if market.upper() == 'US':
        returns_df = _download_us(codes, start_date, end_date)
    else:
        returns_df = _download_a(codes, start_date, end_date)

    if isinstance(returns_df, str):   # 出错时返回错误字符串
        return returns_df

    _portfolio_data['returns'] = returns_df
    _portfolio_data['stocks'] = codes
    _portfolio_data['market'] = market.upper()

    summary = {
        '状态': '下载成功',
        '市场': 'A股' if market.upper() == 'A' else '美股',
        '股票代码': codes,
        '有效交易日数': len(returns_df),
        '日期范围': f'{str(returns_df.index[0])} 至 {str(returns_df.index[-1])}',
        '各股年化收益率（估算）': {
            code: f'{returns_df[code].mean() * 252:.2%}' for code in codes
        },
        '提示': '数据已保存，可调用 markowitz_optimize 进行优化',
    }
    return json.dumps(summary, ensure_ascii=False, indent=2)


def _download_a(codes: list, start_date: str, end_date: str):
    '''A 股数据：akshare 前复权日涨跌幅'''
    dfs = {}
    for code in codes:
        try:
            df = ak.stock_zh_a_hist(
                symbol=code, period='daily',
                start_date=start_date, end_date=end_date,
                adjust='qfq',
            )
            dfs[code] = df.set_index('日期')['涨跌幅'] / 100
        except Exception as e:
            return json.dumps({'错误': f'下载 {code} 失败：{e}'}, ensure_ascii=False)
    return pd.DataFrame(dfs).dropna()


def _download_us(codes: list, start_date: str, end_date: str):
    '''美股数据：yfinance 复权收盘价日收益率'''
    start = pd.to_datetime(start_date, format='%Y%m%d').strftime('%Y-%m-%d')
    end   = pd.to_datetime(end_date,   format='%Y%m%d').strftime('%Y-%m-%d')
    try:
        if len(codes) == 1:
            raw = yf.download(codes[0], start=start, end=end,
                              progress=False, auto_adjust=True)
            close = raw[['Close']].rename(columns={'Close': codes[0]})
        else:
            raw = yf.download(codes, start=start, end=end,
                              progress=False, auto_adjust=True)
            close = raw['Close']
            close.columns = [str(c) for c in close.columns]
        return close.pct_change().dropna()
    except Exception as e:
        return json.dumps({'错误': f'yfinance 下载失败：{e}'}, ensure_ascii=False)


In [10]:
@tool
def markowitz_optimize(risk_free_rate: float = 0.02) -> str:
    '''对已下载的股票数据进行马科维茨投资组合优化。

    同时求解两种组合：
    - 最大夏普比率组合（风险调整后收益最优，适合积极型投资者）
    - 最小方差组合（组合波动率最低，适合保守型投资者）

    请先调用 download_stock_data 下载数据后再使用此工具。

    Args:
        risk_free_rate: 年化无风险利率，默认 0.02（2%）。
                        美股建议使用 0.045（对应近期美国国债收益率）

    Returns:
        两种最优组合的权重及风险收益指标（JSON 字符串）
    '''
    if 'returns' not in _portfolio_data:
        return '错误：请先调用 download_stock_data 下载数据后再调用此工具。'

    returns_df = _portfolio_data['returns']
    stocks = _portfolio_data['stocks']
    n = len(stocks)

    mean_ret = returns_df.mean().values
    cov_mat  = returns_df.cov().values * 252

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0.0, 1.0)] * n
    w0 = np.ones(n) / n

    def neg_sharpe(w):
        ret = float(np.dot(w, mean_ret) * 252)
        vol = float(np.sqrt(w @ cov_mat @ w))
        return -(ret - risk_free_rate) / vol

    def port_vol(w):
        return float(np.sqrt(w @ cov_mat @ w))

    res_sr = minimize(neg_sharpe, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    res_mv = minimize(port_vol,   w0, method='SLSQP', bounds=bounds, constraints=constraints)

    def portfolio_stats(w):
        ret = float(np.dot(w, mean_ret) * 252)
        vol = float(np.sqrt(w @ cov_mat @ w))
        return ret, vol, (ret - risk_free_rate) / vol

    sr_ret, sr_vol, sr_sharpe = portfolio_stats(res_sr.x)
    mv_ret, mv_vol, mv_sharpe = portfolio_stats(res_mv.x)

    result = {
        '最大夏普比率组合（激进型）': {
            '权重配置': {stocks[i]: f'{res_sr.x[i]:.2%}' for i in range(n)},
            '年化预期收益': f'{sr_ret:.2%}',
            '年化波动率（风险）': f'{sr_vol:.2%}',
            '夏普比率': round(sr_sharpe, 4),
        },
        '最小方差组合（保守型）': {
            '权重配置': {stocks[i]: f'{res_mv.x[i]:.2%}' for i in range(n)},
            '年化预期收益': f'{mv_ret:.2%}',
            '年化波动率（风险）': f'{mv_vol:.2%}',
            '夏普比率': round(mv_sharpe, 4),
        },
        '参数说明': f'无风险利率 {risk_free_rate:.1%}，基于 {len(returns_df)} 个交易日历史数据',
    }
    return json.dumps(result, ensure_ascii=False, indent=2)


## 第三步：构建 LangChain 智能体

使用 `create_agent` 将 LLM 和工具组合成智能体。


In [11]:
SYSTEM_PROMPT = '''你是一位专业的股票投资顾问，擅长运用马科维茨现代投资组合理论进行资产配置分析，支持 A 股和美股两个市场。

工作流程：
1. 判断用户要分析的市场（A股 or 美股），调用 download_stock_data 时传入对应的 market 参数
2. 再调用 markowitz_optimize 进行投资组合优化
   - A 股无风险利率建议使用 0.02（国内国债）
   - 美股无风险利率建议使用 0.045（美国国债）
3. 根据优化结果，从"风险收益特征"和"适合人群"两个维度给出配置建议

注意：本分析仅为教学演示，不构成实际投资建议。'''

agent = create_agent(
    model=llm,
    tools=[download_stock_data, markowitz_optimize],
    system_prompt=SYSTEM_PROMPT,
)

print('智能体创建成功！')
print('已注册工具：', [t.name for t in [download_stock_data, markowitz_optimize]])


智能体创建成功！
已注册工具： ['download_stock_data', 'markowitz_optimize']


## 第四步：运行演示

### 演示 A：A 股（国内网络推荐）


In [ ]:
query_a = '''请帮我分析以下3只A股的投资组合配置：
- 平安银行（000001）
- 贵州茅台（600519）
- 宁德时代（300750）

使用2024年全年数据，进行马科维茨投资组合优化，并给出配置建议。'''

result = agent.invoke({'messages': [HumanMessage(content=query_a)]})
print(result['messages'][-1].content)


### 演示 B：美股（境外网络推荐）


In [17]:
query_us = '''请帮我分析以下3只美股的投资组合配置：
- 苹果（AAPL）
- 微软（MSFT）
- 亚马逊
- 阿里巴巴

使用2024年全年数据，进行马科维茨投资组合优化，并给出配置建议。'''

result = agent.invoke({'messages': [HumanMessage(content=query_us)]})
print(result['messages'][-1].content)


以下是基于马科维茨现代投资组合理论的详细分析报告：

---

## 📊 美股投资组合优化分析报告

### 一、各股票2024年表现概览

| 股票 | 年化收益率 |
|:---:|:---:|
| 🍎 苹果 (AAPL) | **33.89%** |
| 💻 微软 (MSFT) | **16.45%** |
| 🛒 亚马逊 (AMZN) | **43.22%** |
| 🏮 阿里巴巴 (BABA) | **21.04%** |

---

### 二、两种最优组合对比

#### 🚀 方案一：最大夏普比率组合（激进型）
> **追求风险调整后收益最大化**

| 资产 | 配置权重 |
|:---:|:---:|
| **苹果 (AAPL)** | **49.75%** ✅ 核心持仓 |
| **微软 (MSFT)** | **44.26%** ✅ 核心持仓 |
| **亚马逊 (AMZN)** | **5.99%** |
| **阿里巴巴 (BABA)** | **0.00%** ❌ 未配置 |

| 指标 | 数值 |
|:---:|:---:|
| 年化预期收益 | **37.25%** |
| 年化波动率（风险） | **19.80%** |
| 夏普比率 | **1.654** ⭐ 非常优秀 |

#### 🛡️ 方案二：最小方差组合（保守型）
> **追求最低波动率**

| 资产 | 配置权重 |
|:---:|:---:|
| **苹果 (AAPL)** | **29.01%** |
| **微软 (MSFT)** | **0.00%** ❌ 未配置 |
| **亚马逊 (AMZN)** | **16.77%** |
| **阿里巴巴 (BABA)** | **54.23%** ✅ 核心持仓 |

| 指标 | 数值 |
|:---:|:---:|
| 年化预期收益 | **22.28%** |
| 年化波动率（风险） | **16.74%** |
| 夏普比率 | **1.062** |

---

### 三、配置建议

#### 📌 风险收益特征

| 维度 | 最大夏普比率组合 | 最小方差组合 |
|:---|:---:|:---:|
| **预期收益** | 高（37.25%） | 中高（22.28%） |
| **波动风险** |